# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import DateType, IntegerType, StringType
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver.erp_customers")

# Read from bronze

In [0]:
try:
    df = spark.table("workspace.bronze.erp_cust_az12_raw")
except Exception as e:
    logger.error(f"Failed to read Bronze table: {e}")
    raise

# Data transformations

## Renaming columns

In [0]:
RENAME_MAP = {
    "CID": "customer_number",
    "BDATE": "birth_date",
    "GEN": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

## Casting data types

In [0]:
df = df.withColumn("birth_date", F.col("birth_date").cast(DateType()))

## Customer id cleanup

In [0]:
df = df.withColumn(
    "customer_number",
    F.when(F.col("customer_number").startswith("NAS"),
           F.substring(F.col("customer_number"), 4, F.length(F.col("customer_number"))))
    .otherwise(F.col("customer_number"))
)

## Birthdate validation

In [0]:
df = df.withColumn(
    "birth_date",
    F.when(F.col("birth_date") > F.current_date(), None)
    .otherwise(F.col("birth_date"))
)

## Gender normalization

In [0]:
df = df.withColumn(
    "gender",
    F.when(F.upper(F.col("gender")).isin("FEMALE", "F"), "Female")
    .when(F.upper(F.col("gender")).isin("MALE", "M"), "Male")
    .otherwise("Unknown")       
)

# Write into Silver Table

In [0]:
try:
    df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customers")
except Exception as e:
    logger.error(f"Failed to write Silver table: {e}")
    raise